In [31]:
from warnings import filterwarnings
filterwarnings('ignore')

```
sigmoid(z) = 1 / (1 + e^(-z))
```
The piece doing the "always positive" work is e^(-z) — exponentiation. No matter how negative or positive z is, e^z (or e^(-z)) is always strictly positive. That's the actual trick, not sigmoid as a whole.

So let's steal just that piece: exponentiate each raw score first, then normalize.

**Building it with real numbers**
```
z_Fail = -2.0
z_Pass = 1.5
z_Distinction = 1.2
```
**Exponentiate** each one:
```
e^(-2.0) = 0.135
e^(1.5)  = 4.482
e^(1.2)  = 3.320
```
Look at that — every single one is now positive, guaranteed, no exceptions. Even the very negative -2.0 became a small positive number (0.135) instead of something broken.

Now normalize — divide each by the sum of all three:
```
sum = 0.135 + 4.482 + 3.320 = 7.937
```
```
p_Fail        = 0.135 / 7.937 = 0.017
p_Pass        = 4.482 / 7.937 = 0.565
p_Distinction = 3.320 / 7.937 = 0.418
```
**Check the sum: 0.017 + 0.565 + 0.418 = 1.000 ✓**

That's it. That's the whole mechanism — and it has a name: **Softmax**.
```
softmax(z_i) = e^(z_i) / Σ e^(z_j)   for all classes j
```
Notice this completely fixes both problems from the one-vs-rest hack:

* **Always sums to exactly 1.00** — not by luck, but structurally: every class's numerator is one term in the exact same denominator sum. They can't not sum to 1.
* **No more ambiguous ties** — Pass (0.565) is now clearly the winner over Distinction (0.418), because they were forced to divide up the same pie instead of each independently claiming "yes, probably me."

With 2 classes, softmax says:
```
p_Fail = e^(z_Fail) / (e^(z_Fail) + e^(z_Pass))
p_Pass = e^(z_Pass) / (e^(z_Fail) + e^(z_Pass))
```
Do a little algebra on p_Pass — divide top and bottom by e^(z_Pass):
```
p_Pass = 1 / (e^(z_Fail - z_Pass) + 1)
       = 1 / (1 + e^(-(z_Pass - z_Fail)))
```
That's exactly sigmoid, just with z = z_Pass - z_Fail instead of a single raw score.

Building Categorical Cross-Entropy With Actual Numbers

Recall our student (hours=7), true label = Pass. One-hot encode it:
```
y = [0, 1, 0]     # Fail, Pass, Distinction
```
And our softmax gave us:
```
p = [0.017, 0.565, 0.418]   # Fail, Pass, Distinction
```
The idea: multiply position-by-position, element-by-element, same "matching spots" pattern from Day 2's dot product.
```
y[0] · log(p[0])  =  0 · log(0.017)  =  0 · (-4.075)  =  0
y[1] · log(p[1])  =  1 · log(0.565)  =  1 · (-0.571)  =  -0.571
y[2] · log(p[2])  =  0 · log(0.418)  =  0 · (-0.872)  =  0
```
Sum them up:
```
0 + (-0.571) + 0 = -0.571
```
Negate it (loss should be positive — bigger mistake = bigger positive number):
```
Loss = -(-0.571) = 0.571
```
Look at what just happened. Exactly like Day 9's BCE — the zeros in y automatically killed off the Fail and Distinction terms. Only the true class (Pass, position 1) survived and contributed to the loss. No if/else needed — the one-hot encoding self-selects, same trick as Day 9, just extended from 2 slots to N slots.

The general formula — Categorical Cross-Entropy:
```
Loss = -Σ y_i · log(p_i)     (sum over all classes i)
```
Averaged across the whole dataset (n samples):
```
CCE = -(1/n) · Σ_samples Σ_classes  y_i · log(p_i)
```
Notice this collapses to Day 9's BCE exactly when there are only 2 classes — same self-selecting mechanism, same shape, just one more class doesn't change the pattern at all.

#### Building SoftmaxRegressionScratch — Piece 1: The softmax() Function

In [32]:
import numpy as np

def softmax(z):
    z_shifted = z - np.max(z)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z)

In [33]:
# test aginst hand computed values
z = np.array([-2.0, 1.5, 1.2])
print(softmax(z))

[0.01705089 0.56464776 0.41830135]


In [34]:
z_big = np.array([1000, 1, 1])
print(softmax(z_big))

[1. 0. 0.]


#### Testing axis=0 vs axis=1

In [35]:
Z = np.array([[-2.0, 1.5, 1.2],    # student 1's raw scores
              [ 3.0, 0.5, -1.0]])  # student 2's raw scores

In [36]:
np.max(Z, axis=0)

array([3. , 1.5, 1.2])

In [37]:
np.max(Z, axis=1)

array([1.5, 3. ])

In [38]:
# maxes = np.max(Z, axis=1)
# print(maxes.shape)
# print(Z - maxes)

In [39]:
# The Fix
maxes = np.max(Z, axis=1, keepdims=True)
print(maxes.shape)   # (2, 1) instead of (2,)

(2, 1)


#### Piece 2, Complete: Batch Softmax

In [40]:
def softmax_batch(Z):
    Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z_shifted)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

In [41]:
Z = np.array([[-2.0, 1.5, 1.2],
              [ 3.0, 0.5, -1.0]])
print(softmax_batch(Z))

[[0.01705089 0.56464776 0.41830135]
 [0.90875992 0.07459556 0.01664452]]


In [42]:
I = np.eye(3)
print(I)

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [43]:
def one_hot(y, n_classes):
    return np.eye(n_classes)[y]

In [44]:
y = np.array([0, 1, 2, 1, 0])
print(one_hot(y, 3))

# [[1. 0. 0.]     <- label 0 (Fail)
#  [0. 1. 0.]     <- label 1 (Pass)
#  [0. 0. 1.]     <- label 2 (Distinction)
#  [0. 1. 0.]     <- label 1 (Pass)
#  [1. 0. 0.]]    <- label 0 (Fail)

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]


#### Piece 4: Categorical Cross-Entropy Loss (Vectorized)
CCE = -(1/n) · Σ_samples Σ_classes  y_i · log(p_i)

In [45]:
y_onehot = np.array([[0, 1, 0],
                      [1, 0, 0]])
p = np.array([[0.017, 0.565, 0.418],
              [0.908, 0.075, 0.017]])

elementwise = y_onehot * np.log(p)
print(elementwise)

# [[ 0.    -0.571  0.   ]     <- student 1: only Pass term survives (matches our earlier hand calc!)
#  [-0.0965  0.     0.   ]]   <- student 2: only Fail term survives

[[-0.         -0.57092955 -0.        ]
 [-0.0965109  -0.         -0.        ]]


In [46]:
test = np.array([[1, 2, 3],
                  [4, 5, 6]])
print(np.sum(test))

21


#### Assembling the Full Loss Function

In [47]:
def categorical_cross_entropy(y_onehot, p):
    n_samples = y_onehot.shape[0]
    loss = -np.sum(y_onehot * np.log(p)) / n_samples
    return loss

In [48]:
y_onehot = np.array([[0, 1, 0],
                      [1, 0, 0]])
p = np.array([[0.017, 0.565, 0.418],
              [0.908, 0.075, 0.017]])

print(categorical_cross_entropy(y_onehot, p))

0.33372022410826996


In [49]:
print(np.log(0.0))

-inf


In [50]:
def categorical_cross_entropy(y_onehot, p, epsilon=1e-15):
    p_clipped = np.clip(p, epsilon, 1 - epsilon)
    n_samples = y_onehot.shape[0]
    loss = -np.sum(y_onehot * np.log(p_clipped)) / n_samples
    return loss

In [51]:
p_broken = np.array([[1e-20, 0.5, 0.5]])
y_test = np.array([[1, 0, 0]])
print(categorical_cross_entropy(y_test, p_broken))

34.538776394910684


#### Assembling fit() — Full Class


In [52]:
class SoftmaxRegressionScratch:
    def __init__(self, learning_rate=0.1, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.loss_history = []

    def fit(self, X, y, n_classes):
        ones = np.ones((X.shape[0], 1))
        X_b = np.hstack([ones, X])
        y_onehot = np.eye(n_classes)[y]

        self.W = np.zeros((X_b.shape[1], n_classes))   # shape (n_features+1, n_classes)

        for i in range(self.n_iterations):
            Z = X_b @ self.W
            predictions = softmax_batch(Z)
            gradient = X_b.T @ (predictions - y_onehot) / X_b.shape[0]
            self.W = self.W - self.learning_rate * gradient

            loss = categorical_cross_entropy(y_onehot, predictions)
            self.loss_history.append(loss)
        return self

    def predict_proba(self, X):
        ones = np.ones((X.shape[0], 1))
        X_b = np.hstack([ones, X])
        return softmax_batch(X_b @ self.W)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

In [53]:
hours = np.array([1, 2, 3, 4, 5, 6, 8, 9, 10])
labels = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])   # 0=Fail, 1=Pass, 2=Distinction

X = hours.reshape(-1, 1)   # shape (9, 1) — matches Day 1's slice-vs-index lesson,
                            # need the column shape, not a flat (9,) array

In [54]:
model = SoftmaxRegressionScratch(learning_rate=0.1, n_iterations=2000)
model.fit(X, labels, n_classes=3)

print("Final loss:", model.loss_history[-1])
print("First loss:", model.loss_history[0])
print("Predictions:", model.predict(X))
print("Actual:     ", labels)

Final loss: 0.1821878449785774
First loss: 1.0986122886681098
Predictions: [0 0 0 1 1 1 2 2 2]
Actual:      [0 0 0 1 1 1 2 2 2]


In [55]:
from sklearn.linear_model import LogisticRegression

sk_model = LogisticRegression()
sk_model.fit(X, labels)

print("sklearn predictions:", sk_model.predict(X))
print("Actual:              ", labels)
print()
print("Scratch weights (W):\n", np.round(model.W, 3))
print()
print("sklearn intercept:", np.round(sk_model.intercept_, 3))
print("sklearn coef:     ", np.round(sk_model.coef_, 3))

sklearn predictions: [0 0 0 1 1 1 2 2 2]
Actual:               [0 0 0 1 1 1 2 2 2]

Scratch weights (W):
 [[ 6.75   0.5   -7.249]
 [-1.613  0.234  1.379]]

sklearn intercept: [ 5.099  1.195 -6.294]
sklearn coef:      [[-1.094]
 [ 0.007]
 [ 1.086]]


#### Homework: Adding a 2nd feature

In [56]:
hours    = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
practice = np.array([1, 2, 1, 5, 6, 9, 3, 8, 9, 10])
labels   = np.array([0, 0, 0, 1, 1, 2, 1, 2, 2, 2])
# 0=Fail, 1=Pass, 2=Distinction

In [58]:
# Step 1 — fit hours-only first, and watch it fail on the traps (proving the traps are real, not hypothetical):

X_1feat = hours.reshape(-1, 1)
model_1feat = SoftmaxRegressionScratch(learning_rate=0.1, n_iterations=2000)
model_1feat.fit(X_1feat, labels, n_classes=3)

print("1-feature predictions:", model_1feat.predict(X_1feat))
print("Actual:                ", labels)

1-feature predictions: [0 0 0 1 1 1 2 2 2 2]
Actual:                 [0 0 0 1 1 2 1 2 2 2]


In [59]:
# Step 2 — Add the 2nd Feature, Scale, Refit

X_2feat = np.column_stack([hours, practice])   # shape (10, 2)

# Scale — same Day 6/9 canyon-prevention step, mandatory before gradient descent
mean = X_2feat.mean(axis=0)
std = X_2feat.std(axis=0)
X_scaled = (X_2feat - mean) / std

model_2feat = SoftmaxRegressionScratch(learning_rate=0.1, n_iterations=2000)
model_2feat.fit(X_scaled, labels, n_classes=3)

print("2-feature predictions:", model_2feat.predict(X_scaled))
print("Actual:                ", labels)
print("Final loss:", model_2feat.loss_history[-1])

2-feature predictions: [0 0 0 1 1 2 1 2 2 2]
Actual:                 [0 0 0 1 1 2 1 2 2 2]
Final loss: 0.03581932869293823


## Interview Q&A — Day 12: Softmax Regression

**Q1. Why can't you just use sigmoid for a 3+ class problem?**

Sigmoid answers one yes/no question at a time. If you train 3 separate sigmoids (one-vs-rest), each one has zero awareness the others exist — their outputs don't sum to 1 (e.g. `0.05 + 0.62 + 0.51 = 1.18`), and you can get ambiguous overlaps where two classes both say "probably me." A real probability distribution over mutually exclusive classes must sum to exactly 1.00, and nothing about independent sigmoids guarantees that.

---

**Q2. What does softmax actually do, mechanically?**

It takes raw, unbounded scores (one per class), exponentiates every one (guaranteeing all positive, via the same "always positive" trick sigmoid uses internally), then divides each by the sum of all of them. That division is what forces every class to compete for the same fixed 100% pie — if one class's score grows, it steals a bigger share of the same shared denominator, automatically shrinking everyone else's share.

---

**Q3. Is softmax a completely different algorithm from sigmoid, or related?**

Directly related — sigmoid is the exact 2-class special case of softmax. If you write out 2-class softmax algebraically and simplify, it reduces precisely to `1/(1+e^-(z_Pass - z_Fail))`, which is sigmoid applied to the *difference* between the two classes' raw scores. Softmax is the general form; sigmoid was always just the shortcut that works when there are only 2 classes.

---

**Q4. Why does softmax need the "subtract the max" numerical stability trick?**

Raw scores can get large (e.g. from real neural network logits), and `e^large_number` overflows a float to `inf`, which then turns division into `nan`. Subtracting the max from every score before exponentiating is mathematically guaranteed not to change the final softmax output (the constant cancels out top and bottom in the algebra), but it makes the *largest* value become `e^0=1` and everything else shrink toward 0 instead of exploding — same answer, numerically safe.

---

**Q5. What is one-hot encoding and why do we need it here?**

It converts a single class label (like `1` for "Pass") into a vector with a `1` in that class's position and `0` everywhere else (e.g. `[0, 1, 0]` for 3 classes). We need it because the loss function needs to compare against a full probability *distribution* (one number per class from softmax), and one-hot is how you represent "the true answer is exactly this one class" in that same vector format.

---

**Q6. Why does Categorical Cross-Entropy use `y * log(p)` instead of something like squared error?**

Two reasons, both direct extensions of Day 9's logic. First, the one-hot `y` vector's zeros automatically kill off every wrong class's term in the sum, leaving only the true class's `log(p)` to matter — no if/else needed. Second, `-log(p)` has no ceiling: as a confident-but-wrong prediction's probability approaches 0, the loss shoots toward infinity, keeping the gradient strong exactly where correction is needed most — the same problem squared error had back on Day 9 (its penalty flattens out near confident mistakes).

---

**Q7. Why is the softmax regression gradient almost identical in form to logistic regression's gradient?**

Because softmax is mathematically sigmoid generalized, the same chain-rule cancellation that simplified Day 9's BCE+sigmoid gradient down to `Xᵀ(predictions - y)` happens again here — it simplifies to `Xᵀ(predictions - y_onehot)`. The only real differences are that `predictions` now comes from `softmax()` instead of `sigmoid()`, `y_onehot` replaces the plain `y`, and the weights are now a full matrix `W` (one column per class) instead of a single vector.

---

**Q8. If a single-feature softmax model gets a data point wrong, does adding more training iterations or tuning the learning rate fix it?**

Not necessarily — depends on *why* it's wrong. If the true pattern genuinely can't be represented by one monotonic feature (e.g. a data point that needs the model to go up, then down, then up again as that one feature increases), no amount of training fixes it, because softmax (like sigmoid) can only trace one smooth increasing/decreasing surface per feature. The real fix is adding the feature that actually carries the missing signal — proven directly in today's homework, where a 2nd feature (practice) resolved two points a 1-feature model could never get right, however long it trained.

---

**Q9. What's the practical difference between the from-scratch model's weights and sklearn's for the same problem?**

The predictions typically match exactly, but the raw weight values usually don't — sklearn's `LogisticRegression` applies L2 regularization by default (same Ridge-style penalty from Day 7), which shrinks its weights toward zero compared to an unregularized from-scratch model. Same final decision boundary, different confidence/aggressiveness in getting there — this was confirmed with real numbers in today's build (scratch weights consistently larger in magnitude than sklearn's, class by class).

---

**Q10. What does `-log(1/n_classes)` tell you, and why is it useful?**

It's the exact loss you'll get from a freshly-initialized (zero-weight) softmax model, since zero weights make every class's raw score `0`, and `softmax([0,0,...,0])` always outputs a uniform `1/n_classes` for every class regardless of the true label. It's a handy sanity check for any future multi-class problem: if your very first training-loss print doesn't roughly match `-log(1/n_classes)`, something is wrong with your initialization or setup before training even begins.